In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from src.data_download import download_data
import json 
import pandas as pd
import pickle
import os

data_dir = Path("../data")

In [2]:
# List the categories to download here
categories = ["All_Beauty"]
reload_data = False

if reload_data:
    for category in categories:
        download_data(category)


In [3]:
for category in categories:
    review_file = f"../data/raw/{category}.jsonl"
    metadata_file = f"../data/raw/meta_{category}.jsonl"

    print(f"Data exploration for {category} dataset:")

    # Read reviews
    reviews = []
    with open(review_file, "r") as f:
        for line in f:
            reviews.append(json.loads(line))

    # Read metadata
    meta = []
    with open(metadata_file, "r") as f:
        for line in f:
            meta.append(json.loads(line))

    print("Reviews count:", len(reviews))
    print("Metadata count:", len(meta))

    print("Review fields:", list(reviews[0].keys()))
    print("Metadata fields:", list(meta[0].keys()))

    print("First review:", reviews[0])
    print("First metadata record:", meta[0])


Data exploration for All_Beauty dataset:
Reviews count: 701528
Metadata count: 112590
Review fields: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
Metadata fields: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']
First review: {'rating': 5.0, 'title': 'Such a lovely scent but not overpowering.', 'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!", 'images': [], 'asin': 'B00YQ6X8EO', 'parent_asin': 'B00YQ6X8EO', 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ', 'timestamp': 1588687728923, 'helpful_vote': 0, 'verified_purchase': Tr

In [4]:
from langchain_core.documents import Document
from src.bm25 import text_tokenizer, build_bm25, bm25_search

tokenized_corpus_file = os.path.join(data_dir, "processed", "tokenized_corpus.pkl")

documents = []
tokenized_corpus = []

metadata_lookup = {
    item["parent_asin"]: item
    for item in meta
}

if os.path.exists(tokenized_corpus_file):
    print("Loading tokenized corpus...")
    
    with open(tokenized_corpus_file, "rb") as f:
        tokenized_corpus = pickle.load(f)

else:
    print("Generating tokenized corpus...")
    
    for review in reviews:
        tokens = text_tokenizer(review["text"])
        tokenized_corpus.append(tokens)

    with open(tokenized_corpus_file, "wb") as f:
        pickle.dump(tokenized_corpus, f)

# Build documents (needed for LangChain BM25)
for tokens, review in zip(tokenized_corpus, reviews):
    processed_text = " ".join(tokens)
    product_metadata = metadata_lookup.get(review["parent_asin"], "")
    combined_text = product_metadata["title"] + " " + processed_text
    documents.append(
        Document(
            page_content=combined_text,
            metadata={"asin": review["parent_asin"],
            "product_review": review["text"],
            "product_title": product_metadata["title"],
            "product_rating": review["rating"]
            }
        )
    )


Loading tokenized corpus...


In [5]:
bm25_index_path = "../data/processed/bm25_index.pkl"
rebuild_bm25 = True

if os.path.exists(bm25_index_path) and not rebuild_bm25:

    print("Loading existing BM25 index...")

    with open(bm25_index_path, "rb") as f:
        bm25 = pickle.load(f)

else:

    print("Building BM25 index...")

    bm25 = build_bm25(documents)

    with open(bm25_index_path, "wb") as f:
        pickle.dump(bm25, f)

Building BM25 index...


In [6]:
# This is how you pass a query to the BM25 search

query = "wireless bluetooth headphones"

results = bm25_search(bm25, documents, query, k=5)

for doc, score in results:
    print("Product Title:", doc.metadata.get("product_title"))
    print("Product Review:", doc.metadata.get("product_review")[:200])
    print("ASIN:", doc.metadata.get("asin"))
    print("Product Rating:", doc.metadata.get("product_rating"))
    print("Retrieval Score:", score)
    print()

Product Title: Wireless Earbuds, 455D Stereo Sound Wireless Headphones Wireless Sport Earbud with Breathing Mini in-Ear Sports Earphones Noise Cancelling Headsets, Bluetooth Earbuds
Product Review: Works fine...
ASIN: B014VTGC9I
Product Rating: 5.0
Retrieval Score: 35.066935752274375

Product Title: Yontune Sleep Headphones Headband Timing Wireless Cozy Band Washable for Running Workout Unique Gifts (Lengthened), Black grey
Product Review: Bluetooth didn’t turn on
ASIN: B0B533WQVC
Product Rating: 1.0
Retrieval Score: 31.791503280805657

Product Title: FENCHILIN Vanity Mirror with Lights Bluetooth Lighted Makeup Mirror Touch Screen Wireless Audio Speaker Dimmable Light Detachable 10X Magnification Rechargable Power (Rose Gold)
Product Review: Love the mirror and light's.  Has bluetooth. Can change mirror different positions.
ASIN: B0769VLLW6
Product Rating: 5.0
Retrieval Score: 21.152462808027963

Product Title: FENCHILIN Vanity Mirror with Lights Bluetooth Lighted Makeup Mirror Touch S

In [7]:
# This is how you pass a query to the semantic search

from src.semantic import semantic_search

query = "wireless bluetooth headphones"

results = semantic_search(documents, query, k=5, sample_size=10000, reload_index=True)

for doc, score in results:
    print("Product Title:", doc.metadata.get("product_title"))
    print("Product Review:", doc.metadata.get("product_review")[:200])
    print("ASIN:", doc.metadata.get("asin"))
    print("Product Rating:", doc.metadata.get("product_rating"))
    print("Retrieval Score:", score)
    print()

Product Title: Earbuds Ear Buds Sport Earbuds Running Earbuds in Ear Headphones Wired Earphones with Microphone Mic Stereo and Volume Control Waterproof Wired Earphone Android Mp3 Players Tablet Laptop 3.5mm Audio
Product Review: Other reviews are not for this product. Does not fit well in ear and doesn't come with any way to change fit/size. Can hardly wear when sitting on the couch so there's no way they'd work for running. 
ASIN: B07KQ4XNDV
Product Rating: 1.0
Retrieval Score: 1.0159402

Product Title: MMUSS Sleep Headphones Headband with Ultra Thin Stereo Speakers.Perfect for Sleeping,Sports,Air Travel,Meditation and Relaxation(Black2)
Product Review: I love these. Very comfortable and great sound quality.
ASIN: B07JQ136H9
Product Rating: 5.0
Retrieval Score: 1.1151774

Product Title: Gaming Headphones Chamvict Xbox One Headset with Stereo Sound Ps4 Headset with Mic Noise Canceling for PS4,PC,Laptop,Cell Phone,Xbox,Find Enemies Before They Find You
Product Review: The quality of th

In [8]:
queries = [
    # Easy (Keyword-based)
    "ultra facial barrier-hydrating cleanser",
    "AM Facial Moisturizing Lotion SPF 30",
    "fit me concealer",
    "cheek heat gel cream blush, face makeup",
    "oil control moisturizing gel-cream"
    # Medium (Semantic-based)
    "something to keep your face moisturized all day",
    "makeup to cover up pimples",
    "comfortable lotion for rigid weather",
    # Complex
    "best sunscreen for scuba diving in tropical regions",
    "what’s the best hair treatment to prevent hair loss",
    "good cleanser for busy working professionals who do not have time"
]

In [45]:
all_results = {}

for query in queries:
    # Retrieve 20 since there might be some duplicates of the same product due to multiple reviews/ description chunks
    bm25_results = bm25_search(bm25, documents, query, k=20)
    semantic_results = semantic_search(
        documents,
        query,
        k=20,
        sample_size=10000,
        reload_index=False
    )

    all_results[query] = {
        "bm25": bm25_results,
        "semantic": semantic_results
    }

In [9]:
test_query = queries[0]
print(all_results[test_query]["bm25"][0])
print(all_results[test_query]["semantic"][0])

(Document(metadata={'asin': 'B00U2VQZC4', 'product_review': 'Good facial cleanser for my sensitive skin', 'product_title': 'Neutrogena Ultra Light Facial Cleansing Oil & Makeup Remover, Non-Comedogenic Face Oil Cleanser to Remove Dirt, Oil, Makeup & Waterproof Mascara, 4 fl. oz', 'product_rating': 5.0}, page_content='Neutrogena Ultra Light Facial Cleansing Oil & Makeup Remover, Non-Comedogenic Face Oil Cleanser to Remove Dirt, Oil, Makeup & Waterproof Mascara, 4 fl. oz good facial cleanser my sensitive skin'), 17.792769631882074)
(Document(id='aa87ed2d-a2c1-435c-82d0-92149fa64ba4', metadata={'asin': 'B09ZDQ626L', 'product_review': 'I like the silicone brush, it’s soft and scrubs without scratching.  I wish it comes off, was kinda weird holding the whole bottle to scrub my face.<br />The cleanser looks and smells like unscented liquid soap.  It is like cleaning my face with foaming hand soap. It left it squeaky clean, which is nice. Takes away all the excess oil and grime and makeup.  M

In [29]:
def deduplicate_results(results, key="asin", top_k=5):
    seen = set()
    unique_results = []

    for r in results:
        value = r.get(key, None)
        if value not in seen:
            seen.add(value)
            unique_results.append(r)
        if len(unique_results) == top_k:
            break

    return unique_results

In [30]:
def format_result(result):
    doc, score = result
    
    metadata = getattr(doc, "metadata", {})
    page_content = getattr(doc, "page_content", "")

    return {
        "title": metadata.get("product_title", "N/A"),
        "asin": metadata.get("asin", "N/A"),
        "score": float(score)
    }

In [31]:
formatted_results = {}

for query, result_dict in all_results.items():
    bm25_formatted = [format_result(r) for r in result_dict["bm25"]]
    semantic_formatted = [format_result(r) for r in result_dict["semantic"]]

    # Show only top 5
    formatted_results[query] = {
        "bm25": deduplicate_results(bm25_formatted, key="asin", top_k=5),
        "semantic": deduplicate_results(semantic_formatted, key="asin", top_k=5),
    }

In [53]:
formatted_results[queries[2]]

{'bm25': [{'title': 'L.a Girl Pro Concealer Hd High-definition Concealer ( Pack of 3 ) Gc973',
   'asin': 'B00HEU0EZK',
   'score': 15.154403746503034},
  {'title': 'Maybelline New York Fit Me Concealer 6.8ml - 25 Medium',
   'asin': 'B007B8UOPK',
   'score': 15.069360719475892},
  {'title': 'LA Girl HD Conceal High Definition Pro Concealer 11 Color Choices (Fawn)',
   'asin': 'B017WPO3B2',
   'score': 12.400374373214541},
  {'title': 'ETUDE Big Cover Skin Fit Concealer PRO (# Neutral Peach) (21AD) | Long-Lasting Closely Adhesive Cover Like Real Skin | Smooth and Perfect Makeup | Hides Dark Circles, Redness',
   'asin': 'B08CVGMFRB',
   'score': 11.550442913511324}],
 'semantic': [{'title': 'Boo-Boo Cover-Up Healing Concealer, Medium, 0.13 Ounce',
   'asin': 'B07FX823ZQ',
   'score': 0.7275570631027222},
  {'title': 'US-LONG-DREAM Professional 15 Color Concealer Camouflage Makeup Palette With a Brush',
   'asin': 'B00VFU35EW',
   'score': 0.8717323541641235}]}

In [51]:
formatted_results[queries[7]]

{'bm25': [{'title': 'Tropical Sands All Natural Biodegradable Water Resistant Sunscreen - SPF 8 - 8 fl Oz - Great for Snorkeling - Reef Safe! by Tropical Sands',
   'asin': 'B0184F8LNK',
   'score': 23.84283661099681},
  {'title': 'Fog Free Shower/Travel Safety Mirror',
   'asin': 'B002JAXW6S',
   'score': 23.03695238797598},
  {'title': 'Honu Sunscreen Superior Sun Protection by Starco Brands - With Patented Spray Wand Technology and Broad Spectrum SPF 50 Allows Sunscreen Coverage to all Hard to Reach Spots (2-Pack)',
   'asin': 'B07MHRJXMS',
   'score': 21.825082255134994},
  {'title': 'Scunci No Slip Grip The Evolution Gel Ponytail Holders - 28 Pcs.',
   'asin': 'B00NPD9SBG',
   'score': 21.698301122035772},
  {'title': 'EKLOEN 12PCS/9PCS/6PCS Multifunctional Headband Magic Mask Scarf',
   'asin': 'B0758FDX1W',
   'score': 20.06589227308382}],
 'semantic': [{'title': 'Tropical Sands All Natural Biodegradable Water Resistant Sunscreen - SPF 8 - 8 fl Oz - Great for Snorkeling - Reef S

In [59]:
def results_to_html(results):
    lines = []
    for i, r in enumerate(results, 1):
        title = r["title"][:90] + "..." if len(r["title"]) > 90 else r["title"]
        lines.append(f"{i}. {title}<br><small>ASIN: {r['asin']} | Score: {r['score']:.3f}</small>")
    return "<br><br>".join(lines)

import pandas as pd

from IPython.display import display, HTML

rows = []

for query, result_dict in formatted_results.items():
    rows.append({
        "Query": query,
        "BM25 Results": results_to_html(result_dict["bm25"]),
        "Semantic Results": results_to_html(result_dict["semantic"])
    })

df = pd.DataFrame(rows)
display(HTML(df.to_html(escape=False, index=False)))
df

Query,BM25 Results,Semantic Results
ultra facial barrier-hydrating cleanser,"1. Neutrogena Ultra Light Facial Cleansing Oil & Makeup Remover, Non-Comedogenic Face Oil Cle...ASIN: B00U2VQZC4 | Score: 17.793","1. GOMAY Hydrating Amino Acid Foaming Facial Cleanser, Daily Face Wash for Makeup Remover wit...ASIN: B09ZDQ626L | Score: 0.6302. REBONCEL Aqua Rich Hydrating Face Foam Cleanser Gentle Hypoallergenic pH Balance Korean Sk...ASIN: B09X23VTSQ | Score: 0.6643. Hylunia Hydrate Body Wash - Energizing Blend With Mango 8.5 ozASIN: B07YNDWRCB | Score: 0.7084. Black Wolf - Men’s Gentle Hydrating Face Wash - 5 Fl Oz - Hydrating Sugar Technology Blend...ASIN: B08GMW3HNG | Score: 0.7165. CELIMAX The Real Sedum Aqua Boosting Essence - with 65.33% Sedum Extract, Hyaluronic Acid ...ASIN: B084WP4XS8 | Score: 0.732"
AM Facial Moisturizing Lotion SPF 30,1. CeraVe AM Facial Moisturizing Lotion SPF 30 | Oil-Free Face Moisturizer with Sunscreen | N...ASIN: B010PJYJIY | Score: 31.762,"1. Fresh Black Tea Age-Delay Lotion Broad Spectrum Sunscreen SPF 20, 1.6 OunceASIN: B00MAYT8GQ | Score: 0.5322. Alpha Hydrox Sheer Silk Moisturizer SPF 15, 1 OunceASIN: B000N9IQLS | Score: 0.6133. Anew Reversalist Complete Renewal Day Lotion SPF 25ASIN: B00JND88A4 | Score: 0.6214. Lesentia Tinted Moisturizer with SPF 31 – Tinted Moisturizer For Face With Spf and Blemish...ASIN: B0855L611L | Score: 0.653"
fit me concealer,1. L.a Girl Pro Concealer Hd High-definition Concealer ( Pack of 3 ) Gc973ASIN: B00HEU0EZK | Score: 15.1542. Maybelline New York Fit Me Concealer 6.8ml - 25 MediumASIN: B007B8UOPK | Score: 15.0693. LA Girl HD Conceal High Definition Pro Concealer 11 Color Choices (Fawn)ASIN: B017WPO3B2 | Score: 12.4004. ETUDE Big Cover Skin Fit Concealer PRO (# Neutral Peach) (21AD) | Long-Lasting Closely Adh...ASIN: B08CVGMFRB | Score: 11.550,"1. Boo-Boo Cover-Up Healing Concealer, Medium, 0.13 OunceASIN: B07FX823ZQ | Score: 0.7282. US-LONG-DREAM Professional 15 Color Concealer Camouflage Makeup Palette With a BrushASIN: B00VFU35EW | Score: 0.872"
"cheek heat gel cream blush, face makeup","1. All Natural Cream Blush for Lip & Cheek Makeup Contouring, Swept AwayASIN: B07Y2CKDDM | Score: 25.3322. [3 Pack] LSxia Liquid Blush Makeup Gifts for Women, Natural Looking Breathable Feel Cream ...ASIN: B09MB5GCXF | Score: 25.1993. [6 Pack] LSxia Liquid Blush Makeup Gifts for Women, Natural Looking Breathable Feel Cream ...ASIN: B0924RRTM9 | Score: 24.9254. [3 Pack] LSxia Liquid Blush for Cheeks, Natural Looking Breathable Feel Cream Blush Lightw...ASIN: B0923614WG | Score: 24.529","1. [moonshot] Cream Paint Lightfit Air 3g - Blackpink Lisa Makeup Kpop Kbeauty Cosmetics, Air...ASIN: B07V5DV9VZ | Score: 0.6492. Too Faced Throwback Metallic Lipstick - Too Too HotASIN: B07Q633P8S | Score: 0.7243. MODE Glide & Glow 3 in 1 Highlighter Creamy Color Makeup Blush Stick, Long Wear Glowing Ra...ASIN: B00IKRKUPA | Score: 0.7294. Peripera Velvet Cheek 0.1 Ounce 009 Emotional Dry LilacASIN: B07GB37GNJ | Score: 0.7305. (3 Pack) NYC Cheek Glow Powder Blush - Riverside RoseASIN: B00I6Q16PS | Score: 0.738"
oil control moisturizing gel-creamsomething to keep your face moisturized all day,"1. Colonial Dames Concentrated Vitamin E Moisturizing Cream 42,000 I.U. for Hydrating & Moist...ASIN: B07TVFCPGP | Score: 24.5372. Botanical Beauty VITAMIN C Moisturizing Face Oil ORGANIC.100% PURE MOISTURE with 20% Vitam...ASIN: B0115YS3OE | Score: 21.3993. 2 Pcs Liquid Foundation 30ml, Moisturizing Highlighting, Matte Oil Control Concealer Found...ASIN: B08L574MBW | Score: 20.6664. Retinol Face Moisturizer Cream Natural Facial moisturizing Cream 1.76 OZ with Ortho-Hydrox...ASIN: B00KCTER3U | Score: 20.2055. 2pcs Green Tea Purifying Clay Mask Stick, Facial Moisturizing, Oil Control, Deep Cleansing...ASIN: B08X75R4JL | Score: 20.043","1. Dr. Au Anti-Aging Face Oil Retinol Serum by Au Natural Skinfood - Promotes Youthful, Glowi...ASIN: B07NPCT6L5 | Score: 0.6682. Jack Black MP 10 Nourishing Oil, 2 Fl OzASIN: B

,Query,BM25 Results,Semantic Results
0,ultra facial barrier-hydrating cleanser,"1. Neutrogena Ultra Light Facial Cleansing Oil & Makeup Remover, Non-Comedogenic Face Oil Cle...<br><small>ASIN: B00U2VQZC4 | Score: 17.793</small>","1. GOMAY Hydrating Amino Acid Foaming Facial Cleanser, Daily Face Wash for Makeup Remover wit...<br><small>ASIN: B09ZDQ626L | Score: 0.630</small><br><br>2. REBONCEL Aqua Rich Hydrating Face Foam Cleanser Gentle Hypoallergenic pH Balance Korean Sk...<br><small>ASIN: B09X23VTSQ | Score: 0.664</small><br><br>3. Hylunia Hydrate Body Wash - Energizing Blend With Mango 8.5 oz<br><small>ASIN: B07YNDWRCB | Score: 0.708</small><br><br>4. Black Wolf - Men’s Gentle Hydrating Face Wash - 5 Fl Oz - Hydrating Sugar Technology Blend...<br><small>ASIN: B08GMW3HNG | Score: 0.716</small><br><br>5. CELIMAX The Real Sedum Aqua Boosting Essence - with 65.33% Sedum Extract, Hyaluronic Acid ...<br><small>ASIN: B084WP4XS8 | Score: 0.732</small>"
1,AM Facial Moisturizing Lotion SPF 30,1. CeraVe AM Facial Moisturizing Lotion SPF 30 | Oil-Free Face Moisturizer with Sunscreen | N...<br><small>ASIN: B010PJYJIY | Score: 31.762</small>,"1. Fresh Black Tea Age-Delay Lotion Broad Spectrum Sunscreen SPF 20, 1.6 Ounce<br><small>ASIN: B00MAYT8GQ | Score: 0.532</small><br><br>2. Alpha Hydrox Sheer Silk Moisturizer SPF 15, 1 Ounce<br><small>ASIN: B000N9IQLS | Score: 0.613</small><br><br>3. Anew Reversalist Complete Renewal Day Lotion SPF 25<br><small>ASIN: B00JND88A4 | Score: 0.621</small><br><br>4. Lesentia Tinted Moisturizer with SPF 31 – Tinted Moisturizer For Face With Spf and Blemish...<br><small>ASIN: B0855L611L | Score: 0.653</small>"
2,fit me concealer,1. L.a Girl Pro Concealer Hd High-definition Concealer ( Pack of 3 ) Gc973<br><small>ASIN: B00HEU0EZK | Score: 15.154</small><br><br>2. Maybelline New York Fit Me Concealer 6.8ml - 25 Medium<br><small>ASIN: B007B8UOPK | Score: 15.069</small><br><br>3. LA Girl HD Conceal High Definition Pro Concealer 11 Color Choices (Fawn)<br><small>ASIN: B017WPO3B2 | Score: 12.400</small><br><br>4. ETUDE Big Cover Skin Fit Concealer PRO (# Neutral Peach) (21AD) | Long-Lasting Closely Adh...<br><small>ASIN: B08CVGMFRB | Score: 11.550</small>,"1. Boo-Boo Cover-Up Healing Concealer, Medium, 0.13 Ounce<br><small>ASIN: B07FX823ZQ | Score: 0.728</small><br><br>2. US-LONG-DREAM Professional 15 Color Concealer Camouflage Makeup Palette With a Brush<br><small>ASIN: B00VFU35EW | Score: 0.872</small>"
3,"cheek heat gel cream blush, face makeup","1. All Natural Cream Blush for Lip & Cheek Makeup Contouring, Swept Away<br><small>ASIN: B07Y2CKDDM | Score: 25.332</small><br><br>2. [3 Pack] LSxia Liquid Blush Makeup Gifts for Women, Natural Looking Breathable Feel Cream ...<br><small>ASIN: B09MB5GCXF | Score: 25.199</small><br><br>3. [6 Pack] LSxia Liquid Blush Makeup Gifts for Women, Natural Looking Breathable Feel Cream ...<br><small>ASIN: B0924RRTM9 | Score: 24.925</small><br><br>4. [3 Pack] LSxia Liquid Blush for Cheeks, Natural Looking Breathable Feel Cream Blush Lightw...<br><small>ASIN: B0923614WG | Score: 24.529</small>","1. [moonshot] Cream Paint Lightfit Air 3g - Blackpink Lisa Makeup Kpop Kbeauty Cosmetics, Air...<br><small>ASIN: B07V5DV9VZ | Score: 0.649</small><br><br>2. Too Faced Throwback Metallic Lipstick - Too Too Hot<br><small>ASIN: B07Q633P8S | Score: 0.724</small><br><br>3. MODE Glide & Glow 3 in 1 Highlighter Creamy Color Makeup Blush Stick, Long Wear Glowing Ra...<br><small>ASIN: B00IKRKUPA | Score: 0.729</small><br><br>4. Peripera Velvet Cheek 0.1 Ounce 009 Emotional Dry Lilac<br><small>ASIN: B07GB37GNJ | Score: 0.730</small><br><br>5. (3 Pack) NYC Cheek Glow Powder Blush - Riverside Rose<br><small>ASIN: B00I6Q16PS | Score: 0.738</small>"
4,oil control moisturizing gel-creamsomething to keep your face moisturized all day,"1. Colonial Dames Concentrated Vitamin E Moisturizing Cream 42,000 I.U. for Hydrating & Moist...<br><small>ASIN: B07TVFCPGP | Score: 24.537</small><br><br>2. Botanical Bea